In [ ]:
import warnings
warnings.filterwarnings("ignore")

import textwrap
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

from focus import (
    Capability,
    DatasetSplit,
    FocusConfig,
    FocusDataset,
    Track,
    download,
    set_config,
)
from focus.data.frame_dataset import FocusFrameDataset
from focus.data.video_dataset import FocusVideoDataset
from focus.preprocessing import FrameExtractorPreprocessor, VideoTimestampOverlayPreprocessor

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "font.family": "serif",
})

In [ ]:
ROOT_DIR = "/projects/datasets_ML/orena/"
DATASET  = "heico"
set_config(FocusConfig(root_dir=ROOT_DIR))

### Data overview

In [ ]:
records = []
for split in (DatasetSplit.TRAIN, DatasetSplit.TEST):
    for track in (Track.FRAME, Track.SEGMENT):
        try:
            ds = FocusDataset(DATASET, split, track)
            for req, ref in ds:
                records.append({
                    "split":               split.value,
                    "track":               track.value,
                    "qID":                 req.qID,
                    "videoID":             req.videoID,
                    "procedure_type":      req.procedure_type,
                    "start_time":          req.start_time,
                    "end_time":            req.end_time,
                    "duration":            req.duration,
                    "question":            req.question,
                    "answer":              ref.answer,
                    "primary_capability": ref.primary.name,
                    "format":              ref._format,
                    "ood":                 ref.ood,
                    "clinical":            ref.clinical,
                })
        except Exception as exc:
            print(f"[skip] {split.value}/{track.value}: {exc}")

df = pd.DataFrame(records)
print(f"Total QA pairs loaded: {len(df):,}")
df.head()


In [ ]:
# ── 5.1 Question counts per split and track ───────────────────────────
pivot = (
    df.pivot_table(index="track", columns="split", values="qID", aggfunc="count", fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot(kind="bar", ax=ax, width=0.6, edgecolor="white", linewidth=0.8)
ax.set_xlabel("Track")
ax.set_ylabel("Number of questions")
ax.set_title("Question counts per track and split")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Split")
plt.tight_layout()
plt.show()

total = pivot.copy()
total.loc["TOTAL"] = total.sum()
print(total)


In [ ]:
# ── 5.2 Procedure type distribution ──────────────────────────────────
splits_present = df["split"].unique().tolist()
fig, axes = plt.subplots(1, len(splits_present), figsize=(7 * len(splits_present), 5))
if len(splits_present) == 1:
    axes = [axes]

for ax, split in zip(axes, splits_present):
    sub = df[df["split"] == split]["procedure_type"].value_counts()
    colors = plt.cm.tab10.colors[:len(sub)]
    sub.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
    ax.set_title(f"Procedure types — {split}")
    ax.set_xlabel("Count")

plt.tight_layout()
plt.show()


In [ ]:
# ── 5.3 Clip duration distribution (segment & procedure tracks) ───────
sub = df[df["track"].isin(["segment", "procedure"])].copy()

# if sub.empty:
#     print("No segment/procedure data available.")
# else:
#     fig, ax = plt.subplots(figsize=(10, 4))
#     for track, grp in sub.groupby("track"):
#         ax.hist(grp["duration"], bins=40, alpha=0.7, label=track, edgecolor="white")
#     ax.set_xlabel("Clip duration (seconds)")
#     ax.set_ylabel("Count")
#     ax.set_title("Duration distribution — segment and procedure tracks")
#     ax.legend()
#     plt.tight_layout()
#     plt.show()

print(sub.groupby("track")["duration"].describe().round(2))


### Capability Taxonomy Analysis

The FOCUS challenge maps each question to one of **15 leaf capabilities** organised under **5 top-level groups**.


In [ ]:
CAPABILITY_GROUPS = {
    "OBJECT_RECOGNITION":  [
        "OBJECT_IDENTIFICATION", "INSTANCE_MATCHING", "OBJECT_ATTRIBUTES",
        "SPATIAL_LOCALIZATION_CAMERA", "SPATIAL_LOCALIZATION_SITUS",
    ],
    "TEMPORAL_GROUNDING":  ["TEMPORAL_LOCALIZATION", "DURATION_ESTIMATION"],
    "AGGREGATION":         ["OBJECT_AGGREGATION", "EVENT_AGGREGATION"],
    "EVENT_UNDERSTANDING": ["FO_INTERACTION_RECOGNITION", "FO_USAGE_PURPOSE", "TEMPORAL_ORDERING"],
    "COMPLEX_REASONING":   ["FUNCTIONAL_REASONING", "CAUSAL_CONSEQUENCE_REASONING", "MULTI_STEP_REASONING"],
}
CAP_TO_GROUP = {leaf: grp for grp, leaves in CAPABILITY_GROUPS.items() for leaf in leaves}

GROUP_COLORS = {
    "OBJECT_RECOGNITION":  "#4C72B0",
    "TEMPORAL_GROUNDING":  "#DD8452",
    "AGGREGATION":         "#55A868",
    "EVENT_UNDERSTANDING": "#C44E52",
    "COMPLEX_REASONING":   "#8172B2",
}

df["capability_group"] = df["primary_capability"].map(CAP_TO_GROUP).fillna("OTHER")
print("Capability group → leaf mapping loaded.")


In [ ]:
# ── 6.1 Leaf-capability distribution ─────────────────────────────────
cap_counts = df["primary_capability"].value_counts()
bar_colors = [GROUP_COLORS.get(CAP_TO_GROUP.get(c, ""), "#888888") for c in cap_counts.index]

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(range(len(cap_counts)), cap_counts.values, color=bar_colors, edgecolor="white")
ax.set_xticks(range(len(cap_counts)))
ax.set_xticklabels(
    [c.replace("_", "\n") for c in cap_counts.index],
    fontsize=5,
)
ax.set_ylabel("Number of questions")
ax.set_title("Primary capability distribution")

legend_patches = [
    mpatches.Patch(color=v, label=k.replace("_", " ").title())
    for k, v in GROUP_COLORS.items()
]
ax.legend(handles=legend_patches, loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── 6.2 Top-level group share per track (stacked bar) ─────────────────
grp_track = df.groupby(["track", "capability_group"]).size().unstack(fill_value=0)
grp_pct   = grp_track.div(grp_track.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5))
grp_pct.plot(
    kind="bar", ax=ax, stacked=True, width=0.6, edgecolor="white", linewidth=0.5,
    color=[GROUP_COLORS.get(c, "#888888") for c in grp_pct.columns],
)
ax.set_ylabel("Share (%)")
ax.set_title("Capability group distribution per track")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(
    title="Capability group",
    labels=[c.replace("_", " ").title() for c in grp_pct.columns],
    bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9,
)
plt.tight_layout()
plt.show()


### Answer Format Distribution

The eight answer formats are: `binary`, `number`, `percentage`, `fo_class`, `open_ended`, `matching`, `multiple_choice`, `time`.


In [ ]:
fmt_counts = df["format"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(
    fmt_counts.values, labels=fmt_counts.index, autopct="%1.1f%%",
    startangle=140, colors=plt.cm.Set2.colors[:len(fmt_counts)],
)
axes[0].set_title("Answer format distribution (overall)")

fmt_track = df.groupby(["track", "format"]).size().unstack(fill_value=0)
fmt_track.plot(
    kind="bar", ax=axes[1], width=0.7, edgecolor="white",
    color=plt.cm.Set2.colors[:len(fmt_track.columns)],
)
axes[1].set_title("Answer format distribution per track")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title="Format", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()
print(fmt_counts.to_string())


## Frame Track

The **frame track** uses a single video frame as visual input.


In [ ]:
# ── 8.1 Load the frame dataset ────────────────────────────────────────
ds_frame_base = FocusDataset(DATASET, DatasetSplit.TRAIN, Track.FRAME)
ds_frame      = FocusFrameDataset(ds_frame_base, stride=1)

print(f"Frame track (train): {len(ds_frame):,} samples")

sample = ds_frame[0]
print("\n─── Request ───────────────────────────────────────────────")
print(f"  qID            : {sample.request.qID}")
print(f"  videoID        : {sample.request.videoID}")
print(f"  procedure_type : {sample.request.procedure_type}")
print(f"  time window    : {sample.request.start_time:.2f}s – {sample.request.end_time:.2f}s  "
      f"(duration: {sample.request.duration:.2f}s)")
print(f"  question       : {sample.request.question}")
print("\n─── Reference ─────────────────────────────────────────────")
print(f"  answer         : {sample.reference.answer}")
print(f"  format         : {sample.reference._format}")
print(f"  primary cap.   : {sample.reference.primary.name}")
print(f"  secondaries    : {[c.name for c in sample.reference.secondaries]}")
print(f"  ood / clinical : {sample.reference.ood} / {sample.reference.clinical}")
print("\n─── FrameSample ───────────────────────────────────────────")
print(f"  base fps       : {sample.base_fps}")
print(f"  effective fps  : {sample.fps}  (stride={1})")
print(f"  frame count    : {len(sample.frame_paths)}")
for p in list(sample.frame_paths)[:3]:
    print(f"  {p}")
if len(sample.frame_paths) > 3:
    print(f"  … (+{len(sample.frame_paths) - 3} more)")


In [ ]:
# ── 8.2 Visualisation helpers ─────────────────────────────────────────

def show_frame_sample(sample, title_prefix: str = "") -> None:
    """Display a single FrameSample: one large frame + Q&A annotation below."""
    paths = [Path(p) for p in sample.frame_paths if Path(p).exists()]
    if not paths:
        print("No frames found on disk — run section 4 (preprocessing) first.")
        return

    fig, (ax_img, ax_txt) = plt.subplots(
        2, 1, figsize=(11, 8),
        gridspec_kw={"height_ratios": [5, 1.4]},
    )
    fig.subplots_adjust(hspace=0.04, left=0.02, right=0.98, top=0.94, bottom=0.02)

    ax_img.imshow(Image.open(paths[0]))
    ax_img.axis("off")
    ax_img.set_title(
        f"{title_prefix}{sample.request.videoID} | {sample.request.procedure_type}",
        fontsize=11, pad=6,
    )

    ax_txt.axis("off")
    q   = textwrap.fill(f"Q: {sample.request.question}", 110)
    a   = f"A: {sample.reference.answer}  [{sample.reference._format}]"
    cap = f"Capability: {sample.reference.primary.name.replace('_', ' ').title()}"
    ax_txt.text(0.01, 0.97, q,   transform=ax_txt.transAxes, fontsize=13, va="top")
    ax_txt.text(0.01, 0.52, a,   transform=ax_txt.transAxes, fontsize=13, va="top",
                color="#2a7a2a", fontweight="bold")
    ax_txt.text(0.01, 0.08, cap, transform=ax_txt.transAxes, fontsize=12, va="top",
                color="#555555")
    plt.show()


def show_frame_format_grid(fmt_label: str, samples: list, cols: int = 4) -> None:
    """Show a grid of single-frame samples for one answer format.

    Each column = one sample: image on top, Q&A text below.
    """
    n = min(cols, len(samples))
    fig, axes = plt.subplots(
        2, n, figsize=(6.5 * n, 8),
        gridspec_kw={"height_ratios": [4.5, 2], "hspace": 0.06, "wspace": 0.04},
        squeeze=False,
    )
    fig.subplots_adjust(left=0.01, right=0.99, top=0.91, bottom=0.02)
    fig.suptitle(f"Format: {fmt_label!r}", fontsize=15, fontweight="bold")

    for col, sample in enumerate(samples[:n]):
        paths = [Path(p) for p in sample.frame_paths if Path(p).exists()]

        ax_img = axes[0][col]
        if paths:
            ax_img.imshow(Image.open(paths[0]))
        else:
            ax_img.text(0.5, 0.5, "frame not on disk", ha="center", va="center",
                        transform=ax_img.transAxes, fontsize=10, color="#888")
        ax_img.axis("off")
        ax_img.set_title(sample.request.procedure_type, fontsize=10, pad=4)

        ax_txt = axes[1][col]
        ax_txt.axis("off")
        q   = textwrap.fill(sample.request.question, 38)
        a   = f"A: {sample.reference.answer}"
        cap = sample.reference.primary.name.replace("_", " ").title()
        ax_txt.text(0.02, 0.97, q,   transform=ax_txt.transAxes, fontsize=12, va="top")
        ax_txt.text(0.02, 0.46, a,   transform=ax_txt.transAxes, fontsize=13, va="top",
                    color="#2a7a2a", fontweight="bold")
        ax_txt.text(0.02, 0.08, cap, transform=ax_txt.transAxes, fontsize=11, va="top",
                    color="#444444")

    plt.show()


In [ ]:
# ── 8.3 Show a grid of examples per answer format ─────────────────────
SAMPLES_PER_FORMAT = 2
format_samples: dict[str, list[int]] = {}

for i, (req, ref) in enumerate(ds_frame_base):
    fmt = ref._format
    if fmt not in format_samples:
        format_samples[fmt] = []
    if len(format_samples[fmt]) < SAMPLES_PER_FORMAT:
        format_samples[fmt].append(i)

# Render one grid per format
for fmt, indices in format_samples.items():
    samples = [ds_frame[i] for i in indices]
    show_frame_format_grid(fmt, samples, cols=SAMPLES_PER_FORMAT)


## 9. Segment Track

The **segment track** provides short video clips (≤ 5 minutes) as visual input.


In [ ]:
# ── 9.1 Load the segment dataset ─────────────────────────────────────
ds_seg_base = FocusDataset(DATASET, DatasetSplit.TRAIN, Track.SEGMENT)
ds_seg      = FocusVideoDataset(ds_seg_base, stride=25)  # stride=25 → ~1 fps from a 25 fps source

print(f"Segment track (train): {len(ds_seg):,} samples")

sample_seg = ds_seg[0]
print("\n─── Request ───────────────────────────────────────────────")
print(f"  qID            : {sample_seg.request.qID}")
print(f"  videoID        : {sample_seg.request.videoID}")
print(f"  procedure_type : {sample_seg.request.procedure_type}")
print(f"  time window    : {sample_seg.request.start_time:.2f}s – {sample_seg.request.end_time:.2f}s  "
      f"(duration: {sample_seg.request.duration:.2f}s)")
print(f"  question       : {sample_seg.request.question}")
print("\n─── Reference ─────────────────────────────────────────────")
print(f"  answer         : {sample_seg.reference.answer}")
print(f"  format         : {sample_seg.reference._format}")
print(f"  primary cap.   : {sample_seg.reference.primary.name}")
print(f"  ood / clinical : {sample_seg.reference.ood} / {sample_seg.reference.clinical}")
print("\n─── VideoSample ───────────────────────────────────────────")
print(f"  base fps       : {sample_seg.base_fps}")
print(f"  effective fps  : {sample_seg.fps}  (stride=25)")
print(f"  video_path     : {sample_seg.video_path}")

# Clean up the temporary clip immediately after inspection
sample_seg.video_path.unlink(missing_ok=True)


In [ ]:
# ── 9.2 Visualisation helper ──────────────────────────────────────────
def seconds_to_hhmmss(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours:02}:{minutes:02}:{secs:02}"

def show_segment_sample(sample, n_keyframes: int = 5) -> None:
    """Render *n_keyframes* equally-spaced frames from a VideoSample plus Q&A annotation.

    The temporary MP4 is deleted after reading.
    """
    if not sample.video_path.exists():
        print("Temporary video file not found.")
        return

    vcap  = cv2.VideoCapture(str(sample.video_path))
    total = int(vcap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0:
        print("Empty video clip.")
        vcap.release()
        sample.video_path.unlink(missing_ok=True)
        return

    frame_indices = np.linspace(0, total - 1, n_keyframes, dtype=int)
    frames = []
    for fi in frame_indices:
        vcap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
        ret, frm = vcap.read()
        if ret:
            frames.append(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
    vcap.release()
    sample.video_path.unlink(missing_ok=True)

    n   = len(frames)
    fig = plt.figure(figsize=(max(13, 6.5 * n), 8))
    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.02, hspace=0.05)
    gs  = gridspec.GridSpec(2, n, height_ratios=[5, 1.6], hspace=0.06, wspace=0.04)

    for col, frm in enumerate(frames):
        ax_img = fig.add_subplot(gs[0, col])
        ax_img.imshow(frm)
        ax_img.axis("off")
        t = sample.request.start_time + frame_indices[col] / max(sample.fps, 1e-6)
        ax_img.set_title(f"t≈{seconds_to_hhmmss(int(t))}", fontsize=20, pad=4)

    ax_txt = fig.add_subplot(gs[1, :])
    ax_txt.axis("off")
    q   = textwrap.fill(f"Q: {sample.request.question}", 130)
    a   = f"A: {sample.reference.answer}  [{sample.reference._format}]"
    cap = f"Capability: {sample.reference.primary.name.replace('_', ' ').title()}"
    ax_txt.text(0.01, 0.96, q,   transform=ax_txt.transAxes, fontsize=20, va="top")
    ax_txt.text(0.01, 0.50, a,   transform=ax_txt.transAxes, fontsize=20, va="top",
                color="#2a7a2a", fontweight="bold")
    ax_txt.text(0.01, 0.08, cap, transform=ax_txt.transAxes, fontsize=20, va="top",
                color="#444444")

    fig.suptitle(
        f"{sample.request.videoID} | {sample.request.procedure_type} | "
        f"{sample.request.duration:.1f}s clip",
        fontsize=20,
    )
    plt.show()


In [ ]:
# ── 9.3 Show one example per answer format ────────────────────────────
# Locate indices using the base dataset first (avoids unnecessary clip extraction)
seg_format_to_idx: dict[str, int] = {}
MAX_FORMATS_SEG = 5
for i, (req, ref) in enumerate(ds_seg_base):
    fmt = ref._format
    if fmt not in seg_format_to_idx:
        seg_format_to_idx[fmt] = i
    if len(seg_format_to_idx) >= MAX_FORMATS_SEG:
        break

for fmt, i in seg_format_to_idx.items():
    print(f"\n{'='*70}")
    req_i, ref_i = ds_seg_base[i]
    print(f"Format: {fmt!r}  |  index: {i}  |  duration: {req_i.duration:.1f}s")
    s = ds_seg[i]
    show_segment_sample(s, n_keyframes=5)


In [ ]:
# ── 9.4 Segment duration statistics ──────────────────────────────────
seg_rows = []
for req, ref in ds_seg_base:
    seg_rows.append({
        "duration":       req.duration,
        "format":         ref._format,
        "capability":     ref.primary.name,
        "procedure_type": req.procedure_type,
    })
ss_df = pd.DataFrame(seg_rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(ss_df["duration"], bins=40, edgecolor="white", color="#55A868")
axes[0].set_xlabel("Duration (s)")
axes[0].set_ylabel("Count")
axes[0].set_title("Segment duration distribution")

ss_df.groupby("format")["duration"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#4C72B0", edgecolor="white",
)
axes[1].set_title("Mean duration by answer format")
axes[1].set_xlabel("Mean duration (s)")

ss_df.groupby("capability")["duration"].median().sort_values().plot(
    kind="barh", ax=axes[2], color="#DD8452", edgecolor="white",
)
axes[2].set_title("Median duration by capability")
axes[2].set_xlabel("Median duration (s)")
axes[2].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()
print(ss_df["duration"].describe().round(2))


## 10. OOD & Clinical Subset Analysis

- **OOD** (`ood=True`): questions that require generalisation beyond the training distribution  
- **Clinical** (`clinical=True`): questions grounded in clinically relevant knowledge


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, flag in zip(axes, ["ood", "clinical"]):
    counts = df.groupby(["track", flag]).size().unstack(fill_value=0)
    counts.plot(kind="bar", ax=ax, width=0.6, edgecolor="white",
                color=["#aaaaaa", "#C44E52"])
    label = "OOD" if flag == "ood" else "Clinical"
    ax.set_title(f"{label} flag distribution per track")
    ax.set_ylabel("Count")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title=label, labels=[f"{label}=False", f"{label}=True"])

plt.tight_layout()
plt.show()

print("OOD counts by track:")
print(df.groupby("track")["ood"].value_counts().unstack(fill_value=0))
print("\nClinical counts by track:")
print(df.groupby("track")["clinical"].value_counts().unstack(fill_value=0))


## 11. Cross-Track Comparison & Summary


In [ ]:
summary = (
    df.groupby(["track", "split"])
    .agg(
        n_questions=("qID",              "count"),
        n_videos=(   "videoID",          "nunique"),
        n_caps=(     "primary_capability","nunique"),
        pct_ood=(    "ood",               "mean"),
        pct_clin=(   "clinical",          "mean"),
        mean_dur=(   "duration",          "mean"),
    )
    .round(3)
)
summary["pct_ood"]  = (summary["pct_ood"]  * 100).round(1)
summary["pct_clin"] = (summary["pct_clin"] * 100).round(1)
summary["mean_dur"] =  summary["mean_dur"].round(1)
summary.columns = ["# Questions", "# Videos", "# Capabilities",
                   "OOD %", "Clinical %", "Mean dur (s)"]
display(summary)


In [ ]:
# ── Capability × track heatmap ────────────────────────────────────────
heat = df.pivot_table(
    index="primary_capability", columns="track",
    values="qID", aggfunc="count", fill_value=0,
)
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(9, max(5, len(heat) * 0.45)))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([c.replace("_", " ").title() for c in heat.index], fontsize=9)

vmax = heat.values.max()
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        color = "white" if v > vmax * 0.55 else "black"
        ax.text(j, i, str(v), ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, label="# Questions")
ax.set_title("Questions per capability × track")
plt.tight_layout()
plt.show()
